# Checking the release receipts against the data

PASSAGE v1.0 ships receipts in `release-receipts/` that state what was generated and verified before
publication. A receipt is only useful if a reader can test it. This notebook tests three of its claims
against the published files:

1. the scene count of every split equals the number of Parquet rows;
2. the cohort and supervision rosters hash to the SHA-256 recorded in the receipt and hold the expected
   number of entries;
3. the outcomes add up: every requested route variant is an exact route.

An optional last section renders the qualification bundle behind the paper's qualification panels,
when one is supplied.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. Hashing the rosters reads about ten gigabytes; set `hash_rosters = False` to skip that step.

In [ ]:
resolutions = [64, 128, 256, 512, 1024, 2048, 4096]
hash_rosters = True  # recompute the SHA-256 of the cohort and supervision rosters
qualification_root = None  # optional qualification bundle for Section 5; None skips that section
output_dir = "notebook-artifacts/notebooks/paper/dataset_qualification"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))




def release_receipt(name):
    """Read a published release receipt from the local copy, or from the pinned Hub revision."""
    if source.root is not None:
        return json.loads((Path(source.root) / "release-receipts" / name).read_text())
    from huggingface_hub import hf_hub_download
    return json.loads(Path(hf_hub_download(dataset_repo, f"release-receipts/{name}", repo_type="dataset",
                                           revision=dataset_revision)).read_text())

## 1. What the receipt states

`release-verification.json` records, for each resolution, the requested scenes, the scenes per split,
the outcome of every route variant and the hashes of two rosters: the cohort roster lists every scene
with its variant statuses; the supervision roster lists every exact route.

In [ ]:
verification = release_receipt("release-verification.json")
SPLITS = ("train", "calibration", "validation", "test")
stated = pd.DataFrame([{
    "Resolution": f"{r}×{r}", "Requested scenes": verification["resolutions"][str(r)]["requested_scenes"],
    "Core scenes": sum(verification["resolutions"][str(r)]["core_counts"].values()),
    "Exact routes": verification["resolutions"][str(r)]["successful_route_variants"],
    "Unresolved": verification["resolutions"][str(r)]["unresolved_variants"]} for r in resolutions])
show(stated, f"Table 1. Release verification receipt: status {verification['status']}, admission "
             f"{verification['admission']['status']}.")

## 2. Check A: scene counts against the Parquet files

The footers of the core Parquet files give the row count of every split without reading the data.

In [ ]:
from passage.notebook_reports import split_inventory_report

footers = {r["resolution"]: r for r in split_inventory_report(source, resolutions)["records"]}
check_a = pd.DataFrame([{
    "Resolution": f"{r}×{r}", "Split": s, "Parquet rows": footers[r][s],
    "Receipt": verification["resolutions"][str(r)]["core_counts"][s],
    "Match": footers[r][s] == verification["resolutions"][str(r)]["core_counts"][s]}
    for r in resolutions for s in SPLITS])
display(Markdown(f"**{int(check_a.Match.sum())} of {len(check_a)}** resolution-split counts match the receipt."))
show(check_a[~check_a.Match] if (~check_a.Match).any() else check_a.groupby("Resolution", sort=False)
     .agg(**{"Parquet rows": ("Parquet rows", "sum"), "Receipt": ("Receipt", "sum"), "Match": ("Match", "all")})
     .reset_index(), "Table 2. Core scenes per resolution: Parquet files against the receipt.")

## 3. Check B: the rosters

Each roster is a JSON-lines file. The cohort roster should hold one line per requested scene, and the
supervision roster one line per exact route. Their SHA-256 must equal the hashes in the receipt.

In [ ]:
import hashlib


def roster_file(name):
    if source.root is not None:
        return Path(source.root) / "release-receipts" / name
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(dataset_repo, f"release-receipts/{name}", repo_type="dataset",
                                revision=dataset_revision))


def digest_and_lines(path):
    sha, lines = hashlib.sha256(), 0
    with open(path, "rb") as handle:
        while chunk := handle.read(1 << 24):
            sha.update(chunk)
            lines += chunk.count(b"\n")
    return sha.hexdigest(), lines


rows = []
for r in resolutions:
    facts = verification["resolutions"][str(r)]
    for kind, expected_hash, expected_lines in (
            ("cohort", facts["cohort_sha256"], facts["requested_scenes"]),
            ("supervision", facts["supervision_sha256"], facts["successful_route_variants"])):
        if not hash_rosters:
            rows.append({"Resolution": f"{r}×{r}", "Roster": kind, "Lines": None, "Expected lines": expected_lines,
                         "Hash matches": None})
            continue
        digest, lines = digest_and_lines(roster_file(f"{kind}-{r}.jsonl"))
        rows.append({"Resolution": f"{r}×{r}", "Roster": kind, "Lines": lines, "Expected lines": expected_lines,
                     "Hash matches": digest == expected_hash})
check_b = pd.DataFrame(rows)
show(check_b, "Table 3. Roster hashes and line counts against the receipt.")

## 4. Check C: the outcomes add up

For every resolution and split, the receipt must count one exact route for each of the six variants of
every scene, and no other search outcome.

In [ ]:
rows = []
for r in resolutions:
    facts = verification["resolutions"][str(r)]
    for s in SPLITS:
        exact = facts["reference_status_counts"].get(f"{s}:exact", 0)
        other = sum(n for key, n in facts["reference_status_counts"].items()
                    if key.startswith(f"{s}:") and key != f"{s}:exact")
        rows.append({"resolution": r, "split": s, "variants": 6 * facts["core_counts"][s], "exact": exact,
                     "other": other, "adds_up": exact == 6 * facts["core_counts"][s] and other == 0})
check_c = pd.DataFrame(rows)
display(Markdown(f"**{int(check_c.adds_up.sum())} of {len(check_c)}** resolution-split cells hold exactly "
                 "six exact routes per scene."))

## 5. The qualification bundle (optional)

The paper's qualification panels come from a separate bundle of receipts: source authentication, the
metadata census, and a stratified route audit with a fixed exact-search budget. The bundle is not
published with the dataset. When `qualification_root` points to a bundle, this section renders its
report and figures; otherwise it records that it was skipped. The audit selects scenes with unequal
probabilities, so its counts describe the selected scenes, not the population.

In [ ]:
if qualification_root is None:
    bundle, bundle_sections = None, {}
    display(Markdown("No bundle supplied (`qualification_root = None`): this section is skipped."))
else:
    from IPython.display import Image
    from passage.notebook_reports import resolve_evidence_path
    from passage.qualification_reports import qualification_report, render_qualification_figures, with_report_figures

    bundle = qualification_report(resolve_evidence_path(qualification_root, project_root))
    for key, expected in {"repo": dataset_repo, "revision": dataset_revision}.items():
        found = bundle["source"].get(key)
        if found is not None and expected is not None and found != expected:
            raise ValueError(f"The bundle describes {key} {found}, not {expected}.")
    completion = bundle["completion"]
    bundle_sections = {
        "source_authentication": "admitted" if completion["source_authentication"] else "not_run",
        "core_census": "admitted" if completion["metadata_census"] else "not_run",
        "qualification": "admitted" if completion["full_campaign"] else "not_run",
        "classical_diagnostics": "admitted" if completion["route_audit"] else "not_run",
        "geographic_support": "not_run",
    }
    display(Markdown(f"Bundle status: **{bundle['status']}**. {bundle['interpretation']}"))
    if bundle["missing_prerequisites"]:
        show(pd.DataFrame({"Missing from the bundle": bundle["missing_prerequisites"]}),
             "Table 4. Bundle files that are missing; the matching panels stay pending.")
    figure_outputs = render_qualification_figures(bundle, output_path)
    bundle = with_report_figures(bundle, figure_outputs, output_path)
    for name in ("fig_qualification.png", "fig_qualification_metadata.png", "fig_qualification_tile_support.png"):
        if name in figure_outputs:
            display(Image(filename=str(output_path / name)))
    write_report({"report_sha256": bundle["binding"]["report_sha256"], "outputs": figure_outputs},
                 output_path, "qualification_figures.json")
    write_report(bundle, output_path, "dataset_qualification-bundle.json")

## Summary

In [ ]:
b_ok = bool(check_b["Hash matches"].all() and (check_b.Lines == check_b["Expected lines"]).all()) if hash_rosters else None
display(Markdown("\n".join([
    f"- Check A: {int(check_a.Match.sum())} of {len(check_a)} split counts equal the Parquet row counts.",
    ("- Check B: every roster hashes to its recorded SHA-256 and holds the expected number of lines."
     if b_ok else "- Check B: " + ("skipped (`hash_rosters = False`)." if b_ok is None else
                                    "at least one roster differs from the receipt; see Table 3.")),
    f"- Check C: {int(check_c.adds_up.sum())} of {len(check_c)} cells hold exactly one exact route per variant.",
    "- Qualification bundle: " + ("not supplied; Section 5 was skipped." if bundle is None
                                  else f"status {bundle['status']}."),
])))
report = {"receipt_status": verification["status"], "stated": stated.to_dict("records"),
          "bundle_status": None if bundle is None else bundle["status"],
          "check_a": check_a.to_dict("records"), "check_b": check_b.to_dict("records"),
          "check_c": check_c.to_dict("records"), "figures": []}

In [ ]:
report_file = write_report(report, output_path, "dataset_qualification.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*